(1) Read the CSV file "notebook/01-venues.csv" containing information about publication venues. Using pandas, and SQLite, load the dataset into a DataFrame ensuring all columns are treated as strings and that missing values are not automatically converted to NaN.

Create a new column called internalId, placed as the first column, whose value is generated from the row index using the format `venue-<index>`.

Then, create a second DataFrame containing the internal identifier and an Italian translation of the venue type: `"journal" > "Rivista"`; `"Book" > "Libro"`.

Finally, store both DataFrames in a SQLite database named.

In [26]:
from pandas import read_csv, Series, DataFrame, read_sql
from sqlite3 import connect

venues = read_csv("../notebook/01-venues.csv", 
                  keep_default_na=False,
                  dtype={
                      "id": "string",
                      "name": "string",
                      "type": "string"
                  })

ita_venues = []
venue_internal_id = []
for idx, row in venues.iterrows():
    vid = "venue-" + str(idx)
    venue_internal_id.append(vid)
    
    if row["type"] == "journal":
        ita_venues.append([vid,"Rivista"])
    else:
        ita_venues.append([vid,"Libro"])
    
venues.insert(0, "internalId", Series(venue_internal_id, dtype="string"))
df_ita_venues = DataFrame(ita_venues, columns=["internalId","typeIta"])

with connect("venues-ex-7_1.db") as con:
    venues.to_sql("Venues", con, if_exists="replace", index=False)
    df_ita_venues.to_sql("VenuesIta", con, if_exists="replace", index=False)

(2) Query the sqlite DB you have just created to retrieve all the Italian typologies of all the journals that contains the "Medicine" in their names

In [27]:
with connect("venues-ex-7_1.db") as con:
    query = """
    SELECT typeIta 
    FROM Venues join VenuesIta ON Venues.internalId = VenuesIta.internalId
    Where Venues.name LIKE '%medicine%';
    """
    df_sql = read_sql(query, con)

# show the content of the result of the query
df_sql  

,typeIta
0,Rivista
